# 과제 LV2. 기존 그래프에 새 대체 API 관계를 추가합니다

**과제 LV1이 준비한 `DEPRECATES` 6건을 유지하고, 새 `REPLACED_BY` 관계만 추가합니다.**  
`assignment_extraction_packet.json`과 기존 DB가 준비되어 있어야 합니다. 없으면 과제 LV1부터 실행하세요.

**실습의 목표**

1. 새 대체 API의 ID를 연결합니다.
2. 기존 관계와 근거를 보존하면서 새 관계만 추가합니다.
3. 계층 조회, 재실행과 적재 이력을 확인합니다.
4. 이번 추가만 되돌리고 기존 관계가 그대로인지 검사합니다.

#### 라이브러리와 Neo4j 연결

`.env`의 접속 정보로 연결하고, `run_cypher`로 쿼리를 실행합니다. 데이터와 결과 파일의 경로도 준비합니다.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
from pprint import pprint
from uuid import uuid4
from urllib.parse import urlsplit

from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase

# 학생용은 현재 폴더, 정답은 한 단계 위 폴더의 자료를 사용합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)


def read_json(name):
    """data 폴더의 JSON 파일을 목록 또는 딕셔너리로 읽습니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """처리 결과를 output 폴더에 한글을 유지해 저장합니다."""
    (output_dir / name).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )


# 현재 작업 폴더부터 상위로 올라가 가장 가까운 .env를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
neo4j_uri = os.environ["NEO4J_URI"]
# driver는 여러 쿼리에서 재사용할 DB 연결 통로입니다. 계정 정보는 출력하지 않습니다.
driver = GraphDatabase.driver(
    neo4j_uri,
    auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]),
)
# 연결 객체 생성만으로 접속 성공이 보장되지 않으므로 지금 서버 접속을 확인합니다.
driver.verify_connectivity()


def run_cypher(query, **params):
    """값을 매개변수로 전달하고 Cypher 결과를 딕셔너리 리스트로 돌려줍니다."""
    # 쿼리마다 세션을 열고 with 블록이 끝나면 닫습니다. driver는 계속 재사용합니다.
    with driver.session() as session:
        # RETURN에서 붙인 별칭이 딕셔너리 키가 되어 파이썬에서 조회할 수 있습니다.
        return [record.data() for record in session.run(query, **params)]


# 주소에 계정 정보가 포함되어 있어도 호스트와 포트만 확인합니다.
connection_address = urlsplit(neo4j_uri)
print("Neo4j 연결 완료. 호스트:", connection_address.hostname, "/ 포트:", connection_address.port)

#### 새 개체의 표준 노드 준비

`entity_type`을 레이블로, `standard_id`를 식별자로 사용합니다. 같은 ID가 있으면 기존 노드를 사용합니다.

In [ ]:
# [제공코드]
def put_nodes(catalog):
    """개체 목록의 타입과 표준 ID로 노드를 찾거나 만듭니다.

    Args:
        catalog: standard_id, entity_type, name, aliases를 담은 개체 목록.
    Returns:
        처리한 개체 수. 이미 있던 노드도 포함합니다.
    """
    rows = run_cypher(
        """
    UNWIND $catalog AS item
    // 원래 타입을 레이블로 사용하고 표준 ID로 같은 개체를 찾습니다.
    MERGE (n:$(item.entity_type) {standard_id: item.standard_id})
    // 기존 이름과 별칭은 덮어쓰지 않습니다.
    ON CREATE SET n.name = item.name, n.aliases = item.aliases
    RETURN count(n) AS processed // 새로 만든 수가 아닌 처리한 개체 수입니다.
    """,
        catalog=catalog,
    )
    return rows[0]["processed"]

#### 출처를 포함한 관계 키 만들기

문서 ID, 주어 ID, 관계, 목적어 ID를 같은 순서로 해시해 `claim_id`를 만듭니다. 근거 문구를 바꿔도 이 네 값이 같으면 같은 키입니다.

In [ ]:
# [제공코드]
def claim_key(row):
    """문서 ID와 표준 ID로 표현한 관계를 저장용 키로 바꿉니다.

    Args:
        row: source_doc_id, subject_id, relation, object_id가 있는 기록.
    Returns:
        같은 문서의 같은 관계이면 동일한 문자열 ID.
    """
    values = [
        row["source_doc_id"],
        row["subject_id"],
        row["relation"],
        row["object_id"],
    ]
    # 네 값을 같은 순서로 문자열로 만든 뒤, 같은 입력에 같은 해시값을 얻습니다.
    text = json.dumps(values, ensure_ascii=False)
    return "claim:" + hashlib.sha256(text.encode("utf-8")).hexdigest()

#### 새 관계를 중복 없이 저장하기

`MERGE`로 같은 관계 키를 찾습니다. `ON CREATE SET`은 처음 만든 관계에만 근거와 묶음 ID를 써서 기존 기록을 유지합니다. 반환값은 신규 생성 수가 아닌 처리 수입니다.

In [ ]:
# [제공코드]
def put_claims(rows, batch_id, schema_version):
    """기존 노드 사이에 문서별 관계와 근거를 중복 없이 저장합니다.

    Args:
        rows: 양 끝 표준 ID를 붙인 추출 기록.
        batch_id: 이번 적재 묶음을 구분하는 ID.
        schema_version: 검사에 사용한 스키마 버전 번호.
    Returns:
        양 끝 노드를 찾아 처리한 기록 수. 기존 관계를 찾은 경우도 셉니다.
    """
    records = []
    for row in rows:
        records.append(dict(row, claim_id=claim_key(row)))
    result = run_cypher(
        """
    UNWIND $rows AS item
    // 먼저 저장해 둔 노드를 타입과 표준 ID로 찾습니다.
    MATCH (s:$(item.subject_type) {standard_id: item.subject_id})
    MATCH (o:$(item.object_type) {standard_id: item.object_id})
    // 같은 문서의 같은 관계이면 기존 관계를 재사용합니다.
    MERGE (s)-[r:$(item.relation) {claim_id: item.claim_id}]->(o)
    // 새 관계에만 출처와 근거를 기록해 이전 문서의 근거를 보존합니다.
    ON CREATE SET r.source_doc_id = item.source_doc_id,
                  r.evidence = item.evidence, r.batch_id = $batch_id,
                  r.schema_version = $schema_version
    RETURN count(r) AS processed // 이미 저장되어 있던 관계도 처리 수에 포함합니다.
    """,
        rows=records,
        batch_id=batch_id,
        schema_version=schema_version,
    )
    return result[0]["processed"]

#### 기존 관계와 새 관계 함께 조회하기

정해 둔 문서와 표준 ID 범위의 관계를 `claim_id` 순서로 읽습니다. 출처별 기록과 근거를 대조할 때 사용합니다.

In [ ]:
# [제공코드]
def read_claims(document_ids, known_ids):
    """지정한 문서들에서 내가 등록한 개체 사이의 관계와 근거를 읽습니다.

    Args:
        document_ids: 조회할 출처 문서 ID 목록.
        known_ids: 조회에 포함할 표준 ID 목록.
    Returns:
        관계와 출처를 담은 딕셔너리의 리스트. claim_id 순서로 정렬합니다.
    """
    return run_cypher(
        """
    MATCH (s)-[r]->(o)
    WHERE r.source_doc_id IN $document_ids AND r.claim_id IS NOT NULL
      // 같은 DB에 다른 실습의 그래프가 있어도 이 실습의 개체만 읽습니다.
      AND s.standard_id IN $known_ids AND o.standard_id IN $known_ids
    RETURN r.claim_id AS claim_id, // 문서별 관계를 구분하는 키입니다.
           s.standard_id AS subject_id, s.name AS subject, // 주어의 표준 ID와 이름입니다.
           type(r) AS relation, // 저장한 관계 타입입니다.
           o.standard_id AS object_id, o.name AS object, // 목적어의 표준 ID와 이름입니다.
           r.source_doc_id AS source_doc_id, // 관계의 출처 문서입니다.
           r.evidence AS evidence, r.batch_id AS batch_id // 인용문과 최초 적재 묶음입니다.
    ORDER BY claim_id
    """,
        document_ids=document_ids,
        known_ids=known_ids,
    )

## 1. 새 API의 ID를 연결합니다

과제 LV1의 파일을 읽습니다. 이 단계에서는 기존 관계를 다시 추출하거나 적재하지 않습니다.

In [ ]:
# [제공코드]

# assignment_extraction_packet.json: 앞 교안에서 만든 새 관계, 원문과 검사 결과입니다.
task_packet = json.loads(
    (output_dir / "assignment_extraction_packet.json").read_text(encoding="utf-8")
)
task_batch = task_packet["rows"]  # 검사 전 새 추출 전체입니다.
task_validated = task_packet["validated"]  # 두 검사를 통과해 ID를 연결할 행입니다.
task_rejected = task_packet["rejected"]  # 교안 01의 기각 사유를 그대로 보존합니다.
task_docs = task_packet["documents"]
task_signatures = task_packet["signatures"]
task_schema_version = task_packet["schema_version"]

# 기존 그래프의 개체 목록입니다. 이 셀에서 기존 관계를 다시 저장하지 않습니다.
task_existing = read_json(task_packet["baseline_file"])
task_catalog = task_existing["catalog"]
task_baseline = task_existing["rows"]
print("파일에서 받은 새 관계:", len(task_batch))
print("검사 통과:", len(task_validated), "/ 검사 기각:", len(task_rejected))
for row in task_validated:
    print("ID를 연결할 관계:", (row["subject"], row["relation"], row["object"]))

### 1-1. 원문으로 확인한 대체 API를 목록에 추가

`assignment_approved_entities.json`을 `task_approved`에 읽고 검토 메모를 확인하세요.  
기존 `task_catalog` 뒤에 이어 `task_expanded_catalog`를 만듭니다. 원래 목록은 수정하지 않습니다.

In [ ]:
# (1) assignment_approved_entities.json을 task_approved에 읽고 review_note를 확인하세요.
# (2) task_catalog + task_approved로 task_expanded_catalog를 만드세요.
# (3) 새 API의 이름, 표준 ID와 보완한 목록의 개수를 출력하세요.
# 여기에 코드를 작성하세요.

### 1-2. 타입과 별칭별 ID 조회 준비

`task_ids_by_name`의 키는 `(entity_type, alias)`, 값은 표준 ID 집합입니다. 모든 별칭을 등록하고 같은 ID의 중복은 없애세요.

In [ ]:
# (1) task_ids_by_name을 빈 딕셔너리로 만드세요.
# (2) expanded_catalog의 각 개체와 그 aliases를 순회하세요.
# (3) (entity_type, alias)를 키로 ID를 집합에 모으세요. 같은 ID는 한 번만 셉니다.
# (4) 첫 통과 관계의 주어와 목적어로 조회한 후보 ID를 출력하세요.
# 여기에 코드를 작성하세요.

### 1-3. 검사 통과 행의 양 끝 ID 연결

`task_validated`의 행을 복사해 주어와 목적어의 ID를 찾으세요.

| 변수 | 담을 행 |
|---|---|
| `task_ready` | 양 끝 ID를 모두 하나로 찾은 행. `subject_id`, `object_id`를 붙임 |
| `task_pending` | 한쪽이라도 ID가 없거나 여러 개인 행. `unresolved_names`에 해당 이름을 기록 |
| `task_rejected` | 과제 LV1에서 기각한 행. 그대로 보존 |

`Styler.applymap`과 `Styler.map`은 대체 관계인 서로 다른 API입니다. 같은 ID로 합치지 않습니다.

In [ ]:
# (1) task_ready와 task_pending을 빈 리스트로 만드세요.
# (2) task_validated의 행을 복사하고, subject와 object 각각의 타입과 이름으로 ID를 찾으세요.
# (3) 후보가 하나면 subject_id 또는 object_id에 기록하고, 없거나 여러 개면 이름을 unresolved에 모으세요.
# (4) 미확정 이름이 있으면 unresolved_names를 붙여 pending에, 없으면 ready에 담으세요.
# (5) 세 건수와 양 끝의 표준 ID, 보류한 이름을 출력하세요. rejected는 앞 교안의 결과를 유지합니다.
# 여기에 코드를 작성하세요.

#### ID 연결 검사

In [ ]:
# [자가채점]
assert task_catalog == task_existing["catalog"], "기존 목록은 수정하지 마세요."
assert task_batch == task_packet["rows"], "새 관계 원본을 유지하세요."
assert task_expanded_catalog == task_catalog + task_approved, (
    "기존 목록 뒤에 검토된 새 API를 이어 붙이세요."
)
assert not task_pending, "검토된 새 API 목록으로 ID를 연결하세요."
assert len(task_validated) == len(task_ready) + len(task_pending)
assert task_rejected == task_packet["rejected"], "과제 LV1의 기각 결과를 유지하세요."
assert all(row["subject_id"] != row["object_id"] for row in task_ready), (
    "원래 API와 대체 API는 서로 다른 개체입니다."
)
assert all(row["relation"] == "REPLACED_BY" for row in task_ready)
print("원래 API와 대체 API를 서로 다른 ID로 연결했습니다.")

## 2. 기존 관계를 유지하고 새 관계를 추가합니다

DB의 기존 `DEPRECATES` 6건을 조회합니다. 새 API 노드만 준비한 뒤 관계를 추가합니다.

In [ ]:
# [제공코드]

# 조회 범위에는 기존 관계의 출처와 이번 추출의 출처를 모두 넣습니다.
task_known_ids = [row["standard_id"] for row in task_expanded_catalog]
task_document_ids = sorted({row["source_doc_id"] for row in task_baseline + task_batch})
task_before = read_claims(task_document_ids, task_known_ids)
print("추가 적재 전 DB 관계:", len(task_before))
for row in task_before:
    print("기존 관계:", row["subject"], "->", row["relation"], "->", row["object"])

# 새로 사용할 타입의 표준 ID에도 고유성 제약을 적용합니다.
for entity_type in sorted({row["entity_type"] for row in task_approved}):
    run_cypher(
        f"CREATE CONSTRAINT IF NOT EXISTS FOR (n:{entity_type}) REQUIRE n.standard_id IS UNIQUE"
    )
print("등록을 확인한 새 개체:", put_nodes(task_approved))

### 2-1. 새 관계 추가와 이전 기록 비교

- `task_batch_id`는 `"task:delta:" + str(uuid4())`로 만드세요.
- `put_claims(task_ready, task_batch_id, task_schema_version)`의 결과를 `task_processed`에 받으세요.
- `read_claims(task_document_ids, task_known_ids)`로 적재 후 결과를 읽으세요. 앞에서 정한 문서 ID와 표준 ID 범위를 그대로 사용합니다.

| 변수 | 자료형과 내용 |
|---|---|
| `task_before` | 앞 셀에서 읽은 적재 전 관계 리스트. 비교 기준으로 유지 |
| `task_after` | 새 관계를 추가한 뒤 읽은 전체 관계 리스트 |
| `task_after_by_id` | 키는 `claim_id`, 값은 `task_after`의 해당 행 전체인 딕셔너리 |
| `task_existing_preserved` | 모든 기존 행이 같은 키의 적재 후 행과 같으면 `True`, 아니면 `False` |

**처리 수와 신규 생성 수를 구분하세요.** `task_processed`는 기존 관계를 재사용한 행도 셉니다.  
실제 증가 수는 `len(task_after) - len(task_before)`입니다. ID뿐 아니라 출처와 근거를 포함한 행 전체를 비교합니다.

In [ ]:
# (1) 이번 묶음 ID를 만들고 task_ready만 put_claims에 넘기세요.
# (2) task_after를 조회하고 전후 관계 수와 증가 수를 출력하세요.
# (3) claim_id를 키, 조회 행 전체를 값으로 하는 task_after_by_id를 만드세요.
# (4) 모든 이전 행이 같은 키의 이후 행과 같은지 task_existing_preserved에 담으세요.
# 여기에 코드를 작성하세요.

#### 기존 관계 보존과 추가 대상 검사

In [ ]:
# [자가채점]
assert task_processed == len(task_ready), "처리 수는 task_ready의 행 수와 같아야 합니다."
assert task_existing_preserved, "기존 행의 ID, 출처와 근거를 포함한 전체 값을 비교하세요."
for row in task_before:
    assert task_after_by_id[row["claim_id"]] == row, "기존 ID와 근거를 덮어쓰지 마세요."
for row in task_ready:
    assert claim_key(row) in task_after_by_id, "적재할 새 관계가 DB에 없습니다."
task_before_keys = {row["claim_id"] for row in task_before}
task_new_keys = {claim_key(row) for row in task_ready} - task_before_keys
assert len(task_after) - len(task_before) == len(task_new_keys)
assert sum(row["relation"] == "DEPRECATES" for row in task_after) == 6
print("기존 6관계와 근거를 유지하고 새 대체 관계를 추가했습니다.")

### 2-2. 같은 입력으로 재실행

같은 세 인수로 `put_claims`를 다시 호출하세요. 처리 수는 `task_replay_processed`, 조회 결과는 `task_replayed`에 담습니다.

In [ ]:
# (1) 같은 세 인수로 put_claims를 호출해 task_replay_processed로 받으세요.
# (2) 같은 문서 ID와 표준 ID 범위로 task_replayed를 조회하고 task_after와 비교하세요.
# 여기에 코드를 작성하세요.

#### 재실행 검사

In [ ]:
# [자가채점]
assert task_replay_processed == task_processed
assert task_replayed == task_after, "관계 수뿐 아니라 ID와 근거도 같아야 합니다."
print("재실행해도 같은 결과입니다.")

## 3. 계층 조회와 처리 이력을 확인합니다

대체 관계의 API를 클래스 소속으로 묶어 조회합니다. 계층은 추출 관계와 별도로 정한 연결입니다.

#### 상위 개념 연결

기존과 대체 API의 소속 클래스에 `BROADER`로 연결합니다.

In [ ]:
# [제공코드]
def put_hierarchy(edges):
    """하위 개념에서 상위 개념으로 향하는 BROADER 관계를 만듭니다.

    Args:
        edges: child_id, parent_id와 상위 개념의 이름, 타입, 정의 이유를 담은 목록.
    Returns:
        처리한 계층 관계 수. 이미 있던 관계도 포함합니다.
    """
    # 상위 개념 노드가 없으면 먼저 만듭니다.
    run_cypher(
        """
    UNWIND $edges AS item
    MERGE (parent:$(item.parent_type) {standard_id: item.parent_id})
    ON CREATE SET parent.name = item.parent_name
    RETURN count(parent) AS processed // 이미 있던 개념도 처리 수에 포함합니다.
    """,
        edges=edges,
    )
    # 상위 개념을 모두 준비한 뒤 하위 개념과 연결합니다.
    rows = run_cypher(
        """
    UNWIND $edges AS item
    // ID로 하위 개념과 상위 개념 노드를 찾습니다.
    MATCH (child {standard_id: item.child_id})
    MATCH (parent {standard_id: item.parent_id})
    // 원문에서 추출한 관계와 구분해, 개념의 상하 관계는 BROADER로 연결합니다.
    MERGE (child)-[r:BROADER]->(parent)
    ON CREATE SET r.note = item.note // 왜 이 계층을 정했는지 남깁니다.
    RETURN count(r) AS processed // 이미 있던 관계도 처리 수에 포함합니다.
    """,
        edges=edges,
    )
    return rows[0]["processed"]

### 3-1. Styler 아래의 대체 관계 찾기

아래 쿼리에서 `BROADER`는 소속 클래스를, `REPLACED_BY`는 교체 방향을 찾습니다. `assignment_hierarchy.json`을 적용하고 같은 쿼리를 다시 실행하세요.

In [ ]:
# [제공코드]

task_top_id = "pandas:api:Styler"
task_broad_query = """
MATCH (api:ApiElement)-[:BROADER*1..]->(:ApiElement {standard_id: $top_id})
MATCH (api)-[r:REPLACED_BY]->(replacement:ApiElement)
WHERE r.claim_id IS NOT NULL AND api.standard_id IN $known_ids
RETURN DISTINCT api.name AS old_api, replacement.name AS new_api // 소속 API의 교체 방향입니다.
ORDER BY old_api
"""
task_before_rows = run_cypher(
    task_broad_query, top_id=task_top_id, known_ids=task_known_ids
)
print("계층 연결 전:", task_before_rows)

#### API 계층 연결과 조회 실행

파일을 `task_hierarchy`로 읽고 `put_hierarchy`의 처리 수를 `task_hierarchy_processed`로 받으세요.  
`run_cypher`에 `task_broad_query`, `top_id=task_top_id`, `known_ids=task_known_ids`를 넘깁니다.  
결과 `task_after_rows`는 **계층 연결 후 조회한 대체 API 쌍의 리스트**입니다. 각 행의 `old_api`, `new_api`를 확인하세요.

In [ ]:
# (1) assignment_hierarchy.json을 task_hierarchy에 읽으세요.
# (2) put_hierarchy의 결과를 task_hierarchy_processed에 담으세요.
# (3) 같은 task_broad_query를 실행해 task_after_rows를 받고 출력하세요.
# 여기에 코드를 작성하세요.

#### 계층 조회 검사

실제로 적재한 대체 관계 중 Styler에 속하는 관계와 비교합니다.

In [ ]:
# [자가채점]
assert task_hierarchy_processed == len(task_hierarchy)
task_expected_pairs = set()
for row in task_ready:
    if row["subject"].startswith("Styler."):
        task_expected_pairs.add((row["subject"], row["object"]))
task_found_pairs = {(row["old_api"], row["new_api"]) for row in task_after_rows}
assert task_found_pairs == task_expected_pairs
print("클래스 계층으로 실제 대체 관계를 찾았습니다.")

#### 문서별 적재 이력

이번 추출 문서의 규칙 버전과 검사 건수를 기록합니다.

In [ ]:
# [제공코드]
def record_state(document, version, ready, pending, rejected, batch_id):
    """문서별 마지막 적재 처리의 버전, 검사 건수와 상태를 기록합니다.

    Args:
        document: 출처 doc_id가 있는 원문.
        version: 검사에 적용한 규칙 버전.
        ready, pending, rejected: 해당 문서의 검사 결과 목록.
        batch_id: 이번 적재 묶음 ID.
    Returns:
        문서 ID, 버전, 적재 상태와 검사 건수의 딕셔너리.
    """
    rows = run_cypher(
        """
    MERGE (d:ProcessingState {doc_id: $doc_id})
    SET d.schema_version = $version, d.status = 'stored', d.batch_id = $batch_id,
        d.accepted = $accepted, d.pending = $pending, d.rejected = $rejected,
        d.processed_at = datetime()
    RETURN d.doc_id AS doc_id, d.schema_version AS schema_version, // 문서와 적용 규칙입니다.
           d.status AS status, // stored는 적재 가능 행의 저장을 마친 상태입니다.
           d.accepted AS accepted, d.pending AS pending, d.rejected AS rejected // 이번 검사 건수입니다.
    """,
        doc_id=document["doc_id"],
        version=version,
        batch_id=batch_id,
        accepted=len(ready),
        pending=len(pending),
        rejected=len(rejected),
    )
    return rows[0]

### 3-2. 처리한 문서의 이력 저장

`ProcessingState.doc_id`에 고유성 제약을 만들고, `task_packet["builder_runs"]`의 각 실행에서 `run["document"]`를 꺼내세요.

1. `source_doc_id`가 현재 문서의 `doc_id`와 같은 행만 세 검사 리스트에서 고릅니다.
2. `record_state`에 문서, `task_schema_version`, 해당 문서의 ready/pending/rejected 리스트, `task_batch_id`를 이 순서로 넘깁니다.
3. 반환한 **상태 딕셔너리**를 `task_storage_states` 리스트에 하나씩 담습니다. 관계가 0건인 문서도 기록합니다.

상태의 `accepted`, `pending`, `rejected`는 해당 문서의 세 검사 건수이며, `status="stored"`는 적재 단계까지 완료했다는 뜻입니다.

In [ ]:
# (1) ProcessingState.doc_id의 고유성 제약을 만들고 task_storage_states를 준비하세요.
# (2) 각 run의 document를 꺼내고 source_doc_id가 같은 ready, pending, rejected 행을 고르세요.
# (3) record_state에 문서, task_schema_version, 세 리스트, task_batch_id를 넘기세요.
# (4) 상태 딕셔너리를 task_storage_states에 모아 출력하세요.
# 여기에 코드를 작성하세요.

#### 처리 이력 검사

In [ ]:
# [자가채점]
assert len(task_storage_states) == len(task_packet["builder_runs"])
for state in task_storage_states:
    assert state["schema_version"] == task_schema_version and state["status"] == "stored"
    for key, rows in [
        ("accepted", task_ready),
        ("pending", task_pending),
        ("rejected", task_rejected),
    ]:
        assert state[key] == sum(row["source_doc_id"] == state["doc_id"] for row in rows)
print("추출한 문서의 추가 적재 이력을 남겼습니다.")

### 3-3. 이번 추가만 되돌리기

현재 묶음에서 새로 만든 관계만 제거합니다. 기존 6관계, 노드와 계층은 유지합니다.

In [ ]:
# [제공코드]

# 이번 batch_id로 처음 만든 관계만 지웁니다. 이전 관계, 노드와 BROADER는 유지합니다.
task_removed = run_cypher(
    """
MATCH ()-[r]->()
WHERE r.batch_id = $batch_id
DELETE r
RETURN count(*) AS removed // 이번 묶음에서 삭제한 관계 수입니다.
""",
    batch_id=task_batch_id,
)[0]
task_restored = read_claims(task_document_ids, task_known_ids)
print("되돌린 관계 수:", task_removed["removed"])
print("남은 기존 관계:", len(task_restored))
print("기존 관계와 근거 복원:", task_restored == task_before)

# 관계가 0건인 문서의 완료 상태도 이번 묶음과 함께 되돌립니다.
task_reverted_states = run_cypher(
    """
MATCH (d:ProcessingState {batch_id: $batch_id})
SET d.status = 'reverted'
RETURN d.doc_id AS doc_id, d.status AS status // 추가 적재를 되돌린 문서입니다.
ORDER BY doc_id
""",
    batch_id=task_batch_id,
)
pprint(task_reverted_states)

#### 되돌리기 검사와 연결 종료

In [ ]:
# [자가채점]
assert task_restored == task_before
assert task_removed["removed"] == len(task_new_keys)
assert all(row["status"] == "reverted" for row in task_reverted_states)
assert sum(row["relation"] == "DEPRECATES" for row in task_restored) == 6
print("기존 관계와 근거를 유지한 채 이번 추가만 되돌렸습니다.")
driver.close()